# Kit — technical evaluation notebook

Drive your deployed Kit agent from Python: direct runtime invocation,
the authenticated REST API, session continuity, and memory inspection.

**Prerequisites:** Kit deployed (see `docs/deployment-guide.md`), AWS
credentials in your environment, `boto3` and `requests` installed.


In [ ]:
import boto3, json, uuid

REGION = "ap-southeast-2"  # your deployment region
STACK = "AgentCore-kit-default"

cfn = boto3.client("cloudformation", region_name=REGION)
outputs = {o["OutputKey"]: o["OutputValue"]
           for o in cfn.describe_stacks(StackName=STACK)["Stacks"][0]["Outputs"]}
RUNTIME_ARN = next(v for k, v in outputs.items() if "RuntimeArn" in k)
API_URL = next(v for k, v in outputs.items() if "ApiUrl" in k)
API_KEY_ID = next(v for k, v in outputs.items() if "ApiKeyId" in k)
print("runtime:", RUNTIME_ARN)
print("api:", API_URL)


## 1. Invoke the runtime directly

`InvokeAgentRuntime` is what the Lambda behind the API calls. Session
ids must be at least 33 characters; reuse one to continue a conversation.


In [ ]:
agentcore = boto3.client("bedrock-agentcore", region_name=REGION)

def invoke(prompt, session_id, actor_id="notebook-user"):
    resp = agentcore.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt, "actor_id": actor_id}).encode(),
    )
    text = []
    for line in resp["response"].iter_lines():
        if line.startswith(b"data: "):
            event = json.loads(line[6:])
            delta = event.get("event", {}).get("contentBlockDelta", {}).get("delta", {}).get("text")
            if delta: text.append(delta)
    return "".join(text)

session = f"nb-{uuid.uuid4()}"
print(invoke("Introduce yourself in one sentence.", session))


## 2. Prove the RAG

These facts exist only in the sample corpus — a correct answer is a
correct retrieval.


In [ ]:
print(invoke("How much is 250g of Marrickville Morning, and what's the equipment return window?", session))


## 3. Session continuity and memory


In [ ]:
print(invoke("Remember: I'm evaluating Kit and my codename for it is Bluegum.", session))


In [ ]:
# Same session — short-term memory
print(invoke("What codename did I give this evaluation?", session))


In [ ]:
# NEW session, same actor — long-term memory (allow a minute or two
# after the message above for extraction to run)
print(invoke("What do you remember about me?", f"nb-{uuid.uuid4()}"))


## 4. The REST API

The same agent through API Gateway — what your applications would call.


In [ ]:
import requests

apigw = boto3.client("apigateway", region_name=REGION)
api_key = apigw.get_api_key(apiKey=API_KEY_ID, includeValue=True)["value"]

r = requests.post(API_URL,
    headers={"x-api-key": api_key},
    json={"prompt": "One-line status check: are you healthy?", "actor_id": "notebook-user"})
print(r.status_code, r.json())


## 5. Inspect the memory store

Raw events and extracted long-term records, straight from AgentCore Memory.


In [ ]:
memory_id = next(v for k, v in outputs.items() if "MemoryKitMemoryId" in k)
resp = agentcore.retrieve_memory_records(
    memoryId=memory_id,
    namespace="/users/notebook-user/facts",
    searchCriteria={"searchQuery": "evaluation codename"},
)
for rec in resp["memoryRecordSummaries"]:
    print(f"[{rec['score']:.2f}]", rec["content"]["text"])


## Where to next

- Swap the corpus: `docs/extending-knowledge-base.md`
- Connect MCP tools: `docs/extending-mcp.md`
- Observability: `docs/extending-observability.md`

Built by [Honest Fox](https://honestfox.com.au).
